In [15]:
import importlib
import src.candidate_processor

importlib.reload(src.candidate_processor)

from src.candidate_processor import CandidateProcessor

In [16]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

from src.data_loader import CandidateDataLoader
from src.candidate_processor import CandidateProcessor
from src.semantic_matcher import SemanticMatcher
from src.jd_parser import JDParser

In [17]:
DATA_PATH = PROJECT_ROOT / "data" / "candidates.jsonl"

loader = CandidateDataLoader(DATA_PATH)
processor = CandidateProcessor()

matcher = SemanticMatcher()
parser = JDParser()

Loading embedding model...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8516.77it/s]


Model loaded.


In [18]:
candidates = loader.load_candidates()

print(len(candidates))

100000


In [19]:
candidate = candidates[0]

profile = processor.process_profile(candidate)
skills = processor.process_skills(candidate)

In [20]:
candidate_text = " ".join(
    [
        profile["headline"],
        profile["summary"],
        skills["skills_text"]
    ]
)

print(candidate_text[:700])

Backend Engineer | SQL, Spark, Cloud Software / data professional with 6.9 years of experience building data pipelines, backend systems, and analytics infrastructure. I'm a backend/data hybrid — Spark, Airflow, SQL warehouses are home territory; I'm building competence on the ML side. My toolkit is solid on the data engineering side — Python, SQL, Spark, Airflow, warehouse design — and I've completed a couple of self-directed ML projects (Kaggle competitions, side projects fine-tuning small models). Interested in transitioning toward more AI/ML-focused work, ideally at a company where I can leverage my existing data-infra skills while learning modern ML practice. Tailwind NLP Image Classific


In [21]:
jd = """
Looking for a Backend Engineer
with Python,
AWS,
SQL,
Cloud experience
and at least 3 years of experience.
"""

In [22]:
score = matcher.similarity(
    jd,
    candidate_text
)

print(score)

0.6193630695343018


In [23]:
scores = []

for candidate in candidates[:100]:
    profile = processor.process_profile(candidate)
    skills = processor.process_skills(candidate)

    candidate_text = " ".join([
        str(profile.get("headline", "")),
        str(profile.get("summary", "")),
        str(skills.get("skills_text", ""))
    ])

    score = matcher.similarity(
        jd,
        candidate_text
    )

    scores.append({
        "candidate_id": candidate["candidate_id"],
        "name": profile.get("name"),
        "score": score
    })

print(f"Scored {len(scores)} candidates")

Scored 100 candidates


In [24]:
ranked = sorted(
    scores,
    key=lambda x: x["score"],
    reverse=True
)

In [25]:
for c in ranked[:10]:
    print(
        c["candidate_id"],
        c["name"],
        round(c["score"], 3)
    )

CAND_0000023 Kavya Nair 0.709
CAND_0000067 Shreya Bose 0.693
CAND_0000044 Vihaan Naidu 0.69
CAND_0000051 Meera Arora 0.677
CAND_0000054 Ved Malhotra 0.661
CAND_0000056 Anjali Sen 0.654
CAND_0000038 Myra Trivedi 0.654
CAND_0000025 Anika Kumar 0.654
CAND_0000072 Diya Sharma 0.65
CAND_0000077 Ved Bhatia 0.65
